# CS F407 - Artificial Intelligence
## Laboratory: Agents - Constructing a Goal-Based Agent using a Large Language Model

**Student:** Eshan Saikia  
**ID:** 2023A7PS1024G

This notebook contains the executable implementation and experimental evidence for the Agents laboratory. The written specification, design discussion, LLM reflection, and final interpretation are in the accompanying report.

## 1. Warehouse Navigation Problem

The warehouse is represented as a two-dimensional grid.

```text
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
```

`S` is the start, `G` is the goal, `#` is an obstacle, and `.` is a free cell. The vehicle can move Up, Down, Left, or Right, with one grid-square change per action.

The agent design used here is:

- **State:** current `(row, column)` position.
- **Goal:** reach the cell containing `G`.
- **Decision component:** breadth-first search.
- **Frontier:** FIFO queue.
- **Memory:** parent/action records and discovered-state records for path reconstruction and duplicate prevention.

The complete problem specification and goal-based justification are in the report.

## 2. LLM-Assisted Implementation

### Exact implementation prompt

> I am implementing the warehouse navigation problem below as a goal-based agent in Python. Represent the warehouse as a two-dimensional grid. The agent starts at `S` and must reach `G`. `#` cells are obstacles, `.` cells are free, and the only actions are Up, Down, Left, and Right; each move changes the position by one cell. Implement a simple, deterministic search-based goal-directed agent that finds a collision-free path, prints the path and its length, and reports failure if no path exists. Store parent/action information so the path can be reconstructed. Explain the chosen search method and why it is appropriate for this environment. Keep the implementation dependency-free and easy to inspect.

The first generated implementation was functionally correct on the supplied map. The final notebook adds explicit validation checks and path visualisation; these additions do not change the search problem or the decision rule.

In [1]:
from collections import deque

WAREHOUSE = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]

# Fixed action order makes the run deterministic.
ACTIONS = [
    ("Up", (-1, 0)),
    ("Down", (1, 0)),
    ("Left", (0, -1)),
    ("Right", (0, 1)),
]

def find_symbol(grid, symbol):
    positions = [
        (r, c)
        for r, row in enumerate(grid)
        for c, value in enumerate(row)
        if value == symbol
    ]
    if len(positions) != 1:
        raise ValueError(f"Expected exactly one {symbol!r}, found {len(positions)}.")
    return positions[0]

def bfs_agent(grid):
    # Breadth-first search over legal 4-neighbour moves.
    rows = len(grid)
    cols = len(grid[0])
    start = find_symbol(grid, "S")
    goal = find_symbol(grid, "G")

    frontier = deque([start])
    parent = {start: None}
    action_taken = {}
    states_expanded = 0

    while frontier:
        current = frontier.popleft()
        states_expanded += 1

        if current == goal:
            break

        for action_name, (dr, dc) in ACTIONS:
            nr, nc = current[0] + dr, current[1] + dc
            neighbour = (nr, nc)

            if (
                0 <= nr < rows
                and 0 <= nc < cols
                and grid[nr][nc] != "#"
                and neighbour not in parent
            ):
                parent[neighbour] = current
                action_taken[neighbour] = action_name
                frontier.append(neighbour)

    if goal not in parent:
        return None, None, states_expanded, start, goal

    states = []
    actions = []
    current = goal

    while current is not None:
        states.append(current)
        if current != start:
            actions.append(action_taken[current])
        current = parent[current]

    states.reverse()
    actions.reverse()
    return states, actions, states_expanded, start, goal

states, actions, expanded, start, goal = bfs_agent(WAREHOUSE)

print("Start:", start)
print("Goal:", goal)
print("Path found:", states is not None)
print("Actions:", actions)
print("Path length:", len(actions))
print("States expanded:", expanded)

Start: (1, 1)
Goal: (1, 19)
Path found: True
Actions: ['Right', 'Right', 'Right', 'Down', 'Right', 'Right', 'Right', 'Up', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right']
Path length: 20
States expanded: 59


In [2]:
# Validate the returned solution against the problem specification.

assert states is not None
assert states[0] == start
assert states[-1] == goal
assert len(actions) == len(states) - 1

for current, action, next_state in zip(states, actions, states[1:]):
    delta = (next_state[0] - current[0], next_state[1] - current[1])
    assert delta == dict(ACTIONS)[action]
    assert WAREHOUSE[next_state[0]][next_state[1]] != "#"

print("Validation checks passed.")
print("Every move is adjacent, legal, and collision-free.")

Validation checks passed.
Every move is adjacent, legal, and collision-free.


In [3]:
# Visualise the returned path using '*' on the free cells.

path_set = set(states)

print("Warehouse with the returned path marked by '*':")
for r, row in enumerate(WAREHOUSE):
    rendered = []
    for c, cell in enumerate(row):
        if (r, c) in path_set and cell not in {"S", "G"}:
            rendered.append("*")
        else:
            rendered.append(cell)
    print("".join(rendered))

Warehouse with the returned path marked by '*':
#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


## 3. Result

The supplied warehouse has a valid path from `S` to `G`. The execution output above records the returned action sequence, path length, and number of states expanded.

BFS was chosen because the environment is finite, deterministic, and every movement has unit cost. It explores states in increasing depth, so the first goal path obtained is a minimum-move path for this formulation.

## 4. Implementation Mapping to the Agent Design

| Concept | Implementation |
|---|---|
| Environment | `WAREHOUSE` grid |
| State | `(row, column)` tuple |
| Actions | `ACTIONS` |
| Transition | Add the action's row/column delta |
| Goal test | `current == goal` |
| Frontier | `deque` |
| Discovered states | `parent` dictionary |
| Path reconstruction | `parent` + `action_taken` |
| Output | path, length, states expanded, validation |

The final written answers and discussion are contained in the accompanying report.

## 5. Submission Note

The executable implementation and measured results are in this notebook. The accompanying report contains the problem formulation, agent design, LLM prompt discussion, Think About It answer, and final reflection.